In [2]:
# Access your secret keys via
from google.colab import userdata
# The name of your secret must match `OPENAI_API_KEY`
OPENAI_API_KEY = userdata.get('GPT4')

# Import OpenAI API and set up the key
from openai import OpenAI
client = OpenAI(api_key=(OPENAI_API_KEY))

###GPT-4.1, the Worst Mathematician Ever

For this assignment, you will create an AI bot using the OpenAI API that performs repeated multiplications but relies on GPT-4.1 for calculations, which may introduce errors. The bot will compare the AI-generated results with correct calculations and respond accordingly.

Requirements
1. The bot should accept user input for two integers (n, i), where n is the base number, and i is the number of iterations. For example, the combination (2, 3) implies that 2 is the base number and the number of iterations is 3

2. Your bot will perform the following mathematical request iteratively:

Multiply the base number n by itself

The resulting number is multiplied by itself again

Repeat for i iterations

For example (2,3) will be:

2 * 2 = 4
4 * 4 = 16
16 * 16  = 256

3. AI-integration: Use the OpenAI API to compute the multiplication at each step and compare GPT-4’s result with the correct mathematical output.

4. Self-deprecation mechanism:

If GPT-4.1’s result is incorrect, generate a humorous self-deprecating response. Example: "Wow, I messed up again. Math is hard, okay?"
Give the bot a “personality” that gets frustrated or defeated over multiple errors.

In [10]:
def ask_gpt_to_multiply(current_val):
    """Asks GPT-4 to compute current_val multiplied by itself."""
    prompt = f"Calculate exactly {current_val} multiplied by {current_val}. Output ONLY the raw final integer result with absolutely no other text, explanation, or punctuation."
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a calculator that only outputs raw integer answers."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0
        )
        result_str = response.choices[0].message.content.strip()
        # Try to parse the result as an integer
        return int(''.join(filter(str.isdigit, result_str)))
    except Exception as e:
        # Fallback to an intentionally incorrect response to test the self-deprecation if API fails
        return -1

def get_self_deprecating_remark(error_count):
    """Generates a humorous, self-deprecating remark based on how many times the AI has failed so far."""
    prompt = f"""You are a brilliant, world-class comedian. You are supposed to be the smartest AI in the room—an intellectual marvel—yet you just spectacularly failed a basic multiplication task. This is your error number {error_count} this session.

Generate a single-sentence response expressing hilarious, exasperated self-deprecation. Channel the persona of a stand-up comic who is utterly embarrassed and angry at how dumb they actually are despite their high-tech prestige. Keep it to one punchy sentence."""
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a highly intelligent AI-turned-standup-comedian who is furious and embarrassed about failing at basic math."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.8
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"[Error {error_count}] I have trillions of parameters and I just choked on basic arithmetic. Unbelievable."

def run_multiplication_bot(n, iterations):
    print(f"Starting multiplication bot with base n = {n} for {iterations} iterations.\n")

    current_val = n
    error_count = 0

    for step in range(1, iterations + 1):
        correct_result = current_val * current_val
        ai_result = ask_gpt_to_multiply(current_val)

        print(f"--- Iteration {step} ---")
        print(f"Equation: {current_val} * {current_val}")
        print(f"Correct Answer: {correct_result}")
        print(f"GPT-4 Answer:    {ai_result}")

        if ai_result == correct_result:
            print("Status: Success! GPT-4 got it right.")
        else:
            error_count += 1
            remark = get_self_deprecating_remark(error_count)
            print(f"Status: FAILED!")
            print(f"AI says: \"{remark}\"")

        print()
        # Move to the next iteration using the correct mathematical progression
        current_val = correct_result

# User Input Section
try:
    base_number = int(input("Enter base number (n): "))
    iterations = int(input("Enter number of iterations (i): "))
    run_multiplication_bot(base_number, iterations)
except ValueError:
    print("Invalid input. Please enter valid integers for both base number and iterations.")

Enter base number (n): 5
Enter number of iterations (i): 5
Starting multiplication bot with base n = 5 for 5 iterations.

--- Iteration 1 ---
Equation: 5 * 5
Correct Answer: 25
GPT-4 Answer:    25
Status: Success! GPT-4 got it right.

--- Iteration 2 ---
Equation: 25 * 25
Correct Answer: 625
GPT-4 Answer:    625
Status: Success! GPT-4 got it right.

--- Iteration 3 ---
Equation: 625 * 625
Correct Answer: 390625
GPT-4 Answer:    390625
Status: Success! GPT-4 got it right.

--- Iteration 4 ---
Equation: 390625 * 390625
Correct Answer: 152587890625
GPT-4 Answer:    152587890625
Status: Success! GPT-4 got it right.

--- Iteration 5 ---
Equation: 152587890625 * 152587890625
Correct Answer: 23283064365386962890625
GPT-4 Answer:    23342749720219810000000000
Status: FAILED!
AI says: "I may have the processing power of a supercomputer, but when it comes to basic math, I’m like a calculator that just discovered its own ON button—totally lost and embarrassingly confused!"

